In [ ]:
# Install required libraries
!pip install sentence-transformers faiss-cpu transformers PyMuPDF

In [ ]:
pip install openai==0.28

In [ ]:
import openai
import os

openai.api_key = "Secret key here"
# Set your OpenAI API Key
#os.environ["OPENAI_API_KEY"] = "Secret key here"  # Replace with your actual API key

In [ ]:
# Import libraries
from google.colab import files
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import faiss
import torch
import numpy as np

# Upload PDF files to Google Colab
uploaded = files.upload()

# Function to extract text from a PDF file using PyMuPDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text += page.get_text()
    return text

# Extract text from all uploaded PDF files
pdf_texts = {}
for pdf_file in uploaded.keys():
    if pdf_file.endswith(".pdf"):
        pdf_texts[pdf_file] = extract_text_from_pdf(pdf_file)

# Combine the extracted text into a single string
full_text = " ".join(pdf_texts.values())

# Split the text into chunks for document retrieval
documents = [full_text[i:i+500] for i in range(0, len(full_text), 500)]  # 500-character chunks
print(f"Number of documents loaded: {len(documents)}")

# Generate embeddings and build FAISS index
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
document_embeddings = embedding_model.encode(documents)
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(document_embeddings))

# Load the pre-trained LLM
model_name = "facebook/opt-350m"  # Use a smaller model suitable for CPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cpu")  # Force CPU usage

#this one is for with GPU - paid version, fyi no more free tier for chatgpt as of Nov 1 2024
# Load LLaMA or OpenLLM
#model_name = "meta-llama/Llama-2-7b-hf"  # Example model (you may need GPU for larger models)
#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name, device_map="gpu")

# Function to retrieve documents
def retrieve_documents(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = [documents[idx] for idx in indices[0]]
    return results

Saving HR-Manual.pdf to HR-Manual (3).pdf
Number of documents loaded: 305
Ask a question about the PDF (type 'exit' to quit): can I be demoted


RateLimitError: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.

In [ ]:
# Function to generate a response using RAG
def generate_rag_answer(query):
    # Retrieve relevant documents
    retrieved_docs = retrieve_documents(query)

    # Combine retrieved documents into context
    context = " ".join(retrieved_docs)

    # Construct the input prompt
    input_prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

    # Call OpenAI's ChatGPT API (GPT-4)
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",  # Use "gpt-4" or "gpt-4o" if preferred
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": input_prompt}
        ],
        max_tokens=300,  # Adjust token limit for response size
        temperature=0.7  # Adjust creativity level (0.2 = factual, 0.9 = creative)
    )

    # Extract and return the model's response
    answer = response["choices"][0]["message"]["content"]
    return answer

# Function to log questions and answers
def log_to_file(question, answer, file_name="questions_answers_log.txt"):
    with open(file_name, "a") as log_file:
        log_file.write(f"Question: {question}\nAnswer: {answer}\n\n")

# Function to address the repeating answer
def remove_repetition(text):
    sentences = text.split(". ")
    unique_sentences = []
    for sentence in sentences:
        if sentence not in unique_sentences:
            unique_sentences.append(sentence)
    return ". ".join(unique_sentences)

# Interactive Loop for Asking Questions
while True:
    user_query = input("Ask a question about the PDF (type 'exit' to quit): ")
    if user_query.lower() == "exit":
        print("Exiting. Goodbye!")
        break
    answer = generate_rag_answer(user_query)
    answer = remove_repetition(answer)
    print(f"Answer: {answer}\n")

    # Log the question and answer to a file
    log_to_file(user_query, answer)